<a href="https://colab.research.google.com/github/BobbepalliRavindrababu/AI_study_Assistant/blob/main/SKillMap_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [35]:
!pip install langchain

In [36]:
from langchain.agents import create_agent

In [37]:
!pip install -U langchain-google-genai

In [38]:
from langchain.chat_models import init_chat_model
from google.colab import userdata

google_api_key=userdata.get("GOOGLE_API_KEY")
model=init_chat_model(
    "google_genai:gemini-2.5-flash",
    temperature=0.0,
    google_api_key=google_api_key
)

In [39]:
!pip install langchain-tavily

In [40]:
from langchain_tavily import TavilySearch
from google.colab import userdata

tavily_api_key=userdata.get("TAVILY_API_KEY")

skill_demand_tool=TavilySearch(
    max_results=5,
    search_depth="advanced",
    tavily_api_key=tavily_api_key
)


Testing Skill Demand Tool


In [41]:
# result=skill_demand_tool.invoke({"query":"Genarative AI Skills demand 2026"})
# print(result)

In [42]:
# import requests
# from google.colab import userdata

# rapidapi_key=userdata.get("RAPIDAPI_KEY")

# url="https://jsearch.p.rapidapi.com/search"

# headers={
#     "X-RapidAPI-Key":rapidapi_key,
#     "X-RapidAPI-Host":"jsearch.p.rapidapi.com"
# }

# querystring={
#     "query":"Genarative AI Skills demand 2026",
#     "page":"1",
#     "num_pages":"1"
# }

Building a Custom Tool


In [43]:
import requests
from langchain.tools import tool
from google.colab import userdata

@tool
def search_jobs(skill: str, location: str) -> list:
  """search for jobs requering a specific skills using JSearch from RapidAPI"""

  rapidapi_key=userdata.get("RAPIDAPI_KEY")

  url="https://jsearch.p.rapidapi.com/search"

  headers={
      "X-RapidAPI-Key":rapidapi_key,
      "X-RapidAPI-Host":"jsearch.p.rapidapi.com"
  }

  querystring={
      "query":f"{skill} in {location}",
      "page":"1",
      "country":"in",
      "employment_types":"INTERN,FULLTIME",
      "job_requirements":"no_experience,under_1_years_experience"
  }

  response=requests.get(url,headers=headers,params=querystring)
  data=response.json()

  jobs=data.get("data",[])
  print(f"Found {len(jobs)} jobs\n")

  result=[]
  for job in jobs:
    result.append(
        {"title":job.get("job_title"),
         "company":job.get("employer_name"),
         "location":job.get("job_city"),
         "apply_link":job.get("apply_link")
        }
    )
  return result

Definig The System Prompt


In [44]:
system_prompt = """You are a Skill-to-Career Mapping assistant that helps students understand skill demand and find matching job opportunities.

You have access to these tools:
- skill_demand_tool: Search for industry demand, salary insights, and career trends
- search_jobs: Find actual job listings requiring specific skills

Help the student by researching the skill they ask about and finding relevant opportunities.

Present results in a clean, readable format with clear sections and proper spacing. Include all job details with apply links. Don't use markdown format."""


Import and create checkpointer instance


In [45]:
from langgraph.checkpoint.memory import InMemorySaver
checkpointer=InMemorySaver()


In [46]:
agent=create_agent(
    model=model,
    tools=[skill_demand_tool,search_jobs],
    system_prompt=system_prompt,
    checkpointer=checkpointer
)

In [47]:
user_query="What's the demand for generative ai in the industry and show me related job openings in India"

Configuring thread_id


In [48]:
config={"configurable":{"thread_id":"1"}}

In [49]:
response=agent.invoke({
    "messages":[{"role":"user","content":user_query}]
},config=config)
print(response["messages"][-1].content)


Found 0 jobs

[{'type': 'text', 'text': 'Generative AI Industry Demand:\n\nThe global generative AI market is experiencing significant growth. It was valued at USD 37.89 billion in 2025 and is projected to reach approximately USD 1,206.24 billion by 2035, demonstrating a compound annual growth rate (CAGR) of 36.97% from 2025 to 2034.\n\nKey drivers for this demand include:\n  The increasing adoption of technologies such as high-resolution image generation, text-to-image conversion, and text-to-video conversion.\n  A growing need across industries to modernize workflows, including automation and remote monitoring.\n  Specific applications are emerging in various sectors:\n    Healthcare: Increasing adoption of 3D printing for developing products like organic molecules and prosthetic limbs.\n    Advanced Manufacturing: Demand for complex designs and the need to reduce component size while improving performance.\n    Content Generation & Management: Widely used for drafting marketing copy